# Step 27b — Intervals for the Step 27 comparisons (CPU, about 30 minutes)

**Inputs.** Mount the **Step 19** output (`step19_logits.npz`) and the **Step 27** output (`step27_ddpm.npz`, `step27_corrected_r2.npz`, `step27_em256_fp32.npz`).

For each run, compared with the test-set reference reweighted to that run's class frequencies, we compute the geometric-mean location ratio and the location scatter. Both get 1,000-resample class-block intervals, from 100 bootstrap medians per pair. We also compute the sampling-noise floor of the class-mass error by simulation.

In [ ]:
import glob, numpy as np, time
from itertools import combinations
from scipy.signal import fftconvolve
rng = np.random.default_rng(271); PAIRS = list(combinations(range(10), 2))
def find(pat): p = glob.glob(f'/kaggle/input/**/{pat}', recursive=True); assert p, f'mount the output containing {pat}'; return p[0]
Z19 = np.load(find('step19_logits.npz')); L_TE, W_TE, SIG = Z19['L_TE'], Z19['W_TE'], Z19['SIG_REC']
F_D = np.bincount(W_TE.argmax(1), minlength=10)/len(W_TE)
G = np.linspace(-4.0, 4.0, 4001); DGG = G[1]-G[0]
def wkde_score(m, w):
    m = np.clip(m, -8, 8); w = np.asarray(w, dtype=np.float64); neff = w.sum()**2/np.maximum((w**2).sum(), 1e-12)
    mu = np.average(m, weights=w); sd = np.sqrt(np.average((m-mu)**2, weights=w)); bw = max(1.5*1.06*sd*neff**(-1/5), 1e-3)
    h, _ = np.histogram(m, bins=np.append(G-DGG/2, G[-1]+DGG/2), weights=w, density=True); k = np.exp(-G**2/(2*bw**2)); k /= k.sum()
    p = np.maximum(fftconvolve(h, k, 'same'), 1e-300); return np.gradient(np.log(p), G), p
def _merge(pts, merge=0.03):
    ch = True
    while ch and len(pts) > 1:
        ch = False
        for a in range(len(pts)-1):
            if abs(pts[a][0]-pts[a+1][0]) < merge and pts[a][1] != pts[a+1][1]: del pts[a:a+2]; ch = True; break
    return pts
def stationary(sm, p, lo=-3.5, hi=3.5, prominence=0.05):
    keep = (G > lo) & (G < hi) & (p > 1e-3*p.max()); idx = np.where((np.sign(sm[:-1]) != np.sign(sm[1:])) & keep[:-1] & keep[1:])[0]
    pts = _merge([(float(G[i]-sm[i]*DGG/(sm[i+1]-sm[i])) if sm[i+1] != sm[i] else float(G[i]), 'max' if sm[i] > 0 else 'min') for i in idx])
    logp = np.concatenate([[0.0], np.cumsum(0.5*(sm[1:]+sm[:-1])*DGG)]); lp = lambda x: float(np.interp(x, G, logp)); ch = True
    while ch and len(pts) > 1:
        ch = False
        for a in range(len(pts)-1):
            if pts[a][1] != pts[a+1][1] and abs(lp(pts[a][0]) - lp(pts[a+1][0])) < prominence: del pts[a:a+2]; ch = True; break
    return pts
def n_max(pts): return sum(k == 'max' for _, k in pts)
def cusp(sm, p, window=1.0, b_min=0.02, s_max=0.6):
    ok = (np.abs(G) < window) & (p > 0); c3, c2, c1, c0 = np.polyfit(G[ok], sm[ok], 3, w=np.sqrt(p[ok])); b = max(-c3, b_min); c3 = -b; s = float(np.clip(-c2/(3*c3), -s_max, s_max))
    return c0 + c1*s + c2*s**2 + c3*s**3, c1 + 3*c3*s**2 + 2*c2*s, b
def std_stats(z, wa, wb):
    mua, mub = np.average(z, weights=wa), np.average(z, weights=wb); half = 0.5*(mua-mub); sda = np.sqrt(np.average((z-mua)**2, weights=wa)); sdb = np.sqrt(np.average((z-mub)**2, weights=wb))
    return 0.5*(mua+mub), half, abs(half)/(0.5*(sda+sdb))
STD = {(a, b): [std_stats(L_TE[g, :, a].astype(np.float64) - L_TE[g, :, b], W_TE[:, a] + 1e-12, W_TE[:, b] + 1e-12) for g in range(len(SIG))] for a, b in PAIRS}
def analyse(L, W, a, b, idx, balance=False, rw=None, g0=0, slope=False):
    """Along the pair log-odds (reference standardisation): returns dict(sigma, gi, h, a, b[, s]). rw: extra class reweighting factors (for part C)."""
    Wa = W[idx, a].astype(np.float64); Wb = W[idx, b].astype(np.float64)
    if rw is not None: Wa = Wa*rw[a]; Wb = Wb*rw[b]
    if balance: tot = 0.5*(Wa.sum()+Wb.sum()); Wa = Wa/Wa.sum()*tot; Wb = Wb/Wb.sum()*tot
    def dist(g):
        mid, half, sep = STD[(a, b)][g]
        if sep < 0.3: return None
        z = L[g, idx, a].astype(np.float64) - L[g, idx, b]; return wkde_score((z - mid)/half, Wa + Wb)
    prev = None if g0 == 0 else 1
    for g in range(g0, len(SIG)):
        d = dist(g); nm = 1 if d is None else n_max(stationary(*d))
        if nm >= 2 and prev is not None:
            h, a_, b_ = cusp(*d); out = dict(sigma=float(SIG[g]), gi=g, h=h, a=a_, b=b_)
            if slope:                                            # a(log sigma) near the transition, from the levels just above and below
                gs = [k for k in range(max(0, g-4), min(len(SIG), g+3)) if dist(k) is not None]
                av = [cusp(*dist(k))[1] for k in gs]
                if len(gs) >= 3: out['s'] = -np.polyfit(np.log(SIG[gs]), av, 1)[0]
            return out
        if nm == 1: prev = 1
    return None
def boot(L, W, a, b, n, B, balance=False, ref=False, g_hint=None):
    """Bootstrap medians and 95% intervals of sigma_bif and h at sample size n."""
    ss, hs = [], []; g0 = 0 if g_hint is None else max(0, g_hint - 8)
    for _ in range(B):
        idx = rng.choice(len(W), size=n, replace=True); r = analyse(L, W, a, b, idx, balance, g0=g0)
        if r: ss.append(r['sigma']); hs.append(r['h'])
    if len(ss) < B//3: return None
    return dict(sigma=np.median(ss), s_lo=np.percentile(ss, 2.5), s_hi=np.percentile(ss, 97.5), h=np.median(hs), h_lo=np.percentile(hs, 2.5), h_hi=np.percentile(hs, 97.5), n_ok=len(ss))
def class_block_ci(x, y, pairs, B=5000, fn=None):
    """Class-block bootstrap: resample the 10 classes with replacement, weight each pair by the product of its classes' multiplicities."""
    P = np.array(pairs); out = []
    for _ in range(B):
        cls = rng.choice(10, 10, replace=True); w = np.array([np.sum(cls == a)*np.sum(cls == b) for a, b in P], float)
        if w.sum() < 6: continue
        if fn is None:
            mx, my = np.average(x, weights=w), np.average(y, weights=w); out.append(np.sum(w*(x-mx)*(y-my))/np.sqrt(np.sum(w*(x-mx)**2)*np.sum(w*(y-my)**2)))
        else: out.append(fn(w))
    out = np.array(out, float); out = out[np.isfinite(out)]
    return np.percentile(out, [2.5, 97.5]) if len(out) > 20 else np.array([np.nan, np.nan])

def bmed(L, W, a, b, n, B=100, rw=None):
    r0 = analyse(L, W, a, b, rng.choice(len(W), size=n, replace=len(W) < n), False, rw)
    if r0 is None: return None
    out = []
    for _ in range(B):
        r = analyse(L, W, a, b, rng.choice(len(W), size=n, replace=True), False, rw, g0=max(0, r0['gi'] - 8))
        if r: out.append((r['sigma'], r['h']))
    if len(out) < B//3: return None
    o = np.array(out); return np.median(o[:, 0]), np.median(o[:, 1])
for name, fn in [('DDPM (Ho et al.), EM-256', 'step27_ddpm.npz'), ('EDM, corrected tilt round 2', 'step27_corrected_r2.npz'), ('EDM, EM-256 fp32', 'step27_em256_fp32.npz')]:
    z = np.load(find(fn)); L, W = z['L'], z['W']; n = len(W); RW = (np.bincount(W.argmax(1), minlength=10)/n)/F_D; t0 = time.time(); rows = {}
    for p in PAIRS:
        m = bmed(L, W, *p, n); d = bmed(L_TE, W_TE, *p, n, rw=RW)
        if m and d: rows[p] = (m, d)
    P_ = list(rows); lr = np.array([np.log(rows[p][0][0]/rows[p][1][0]) for p in P_]); hm = np.array([rows[p][0][1] for p in P_]); hd = np.array([rows[p][1][1] for p in P_])
    gm = class_block_ci(None, None, P_, B=1000, fn=lambda w: np.exp(np.average(lr, weights=w)))
    sd = class_block_ci(None, None, P_, B=1000, fn=lambda w: np.sqrt(np.average((lr - np.average(lr, weights=w))**2, weights=w)))
    hc = class_block_ci(hm, hd, P_, B=1000)
    print(f"{name:>30} (n={n}, {len(P_)} pairs, {(time.time()-t0)/60:.0f} min): vs data reweighted to its class mass: geometric-mean location ratio {np.exp(lr.mean()):.3f} [{gm[0]:.3f}, {gm[1]:.3f}]; "
          f"location SD {lr.std():.3f} [{sd[0]:.3f}, {sd[1]:.3f}]; corr h {np.corrcoef(hm, hd)[0,1]:.3f} [{hc[0]:.2f}, {hc[1]:.2f}]")
for n in [1000, 1500, 2000, 2500, 3000, 3500, 10000]:
    sims = [np.mean(np.abs(np.log(np.maximum(np.bincount(rng.choice(10, n), minlength=10)/n, 1e-4)/0.1))) for _ in range(2000)]
    print(f"class-mass error from sampling noise alone (uniform classes), n={n}: mean {np.mean(sims):.3f}, 95th percentile {np.percentile(sims, 95):.3f}")